# 🌸 Marigold V2 360° Panorama Depth Estimation & 3D Reconstruction

GPU-accelerated, production-ready inference notebook for **[marigold_cli](https://github.com/A511-git/marigold_cli)**.

### 🌟 Pipeline Features
- **Pure Marigold V2 DiT Architecture**: Powered by Qwen-Image-Edit-2509 Diffusion Transformer backbone + Huawei Bayer Lab single-step flow matching.
- **4-bit BitsAndBytes Quantization**: Fits comfortably within standard 12GB–16GB Kaggle GPUs (T4 / P100 / L4).
- **12-Camera Icosahedral Splitting**: Seamlessly decomposes 360° equirectangular panoramas into perspective tiles.
- **GPU PyTorch Poisson Blending**: Solves overdetermined multi-scale linear depth stitching in ~4.6s directly on CUDA.
- **Full Production Export**: Outputs `depth.npy` (Float32), `depth.exr` (HDR), `depth_vis.png` (Colorized Turbo), `mask.png`, `normal_vis.png`, and `custom_d2p.ply` (3D Point Cloud).

In [ ]:
# =============================================================================
# 1. CONFIGURATION & PARAMETERS
# =============================================================================
import os
import torch

# ---- Input & Output Paths ----
INPUT_DIR   = "/kaggle/input/datasets/newmailserver/panorama-imgs/panno"   # dataset attached to notebook
WORK_DIR    = "/kaggle/working/marigold_cli"                              # where marigold_cli repo gets cloned
OUTPUT_DIR  = "/kaggle/working/output"                                    # root output folder

# ---- Git Repository & Model Configuration ----
MARIGOLD_REPO       = "https://github.com/A511-git/marigold_cli"
MARIGOLD_CHECKPOINT = "huawei-bayerlab/marigold-v2-0"  # Local path or HuggingFace repo
BASE_MODEL          = "Qwen/Qwen-Image-Edit-2509"     # Local path or HuggingFace repo
MODALITY            = "depth"                         # Options: depth, normals, albedo
QUANTIZATION        = "4bit"                          # Options: 4bit, 8bit, none

# ---- Inference Settings ----
USE_FP16            = True     # Use FP16/BF16 half precision for fast inference
SPLIT_RESOLUTION    = 512      # Resolution per perspective tile (512 or 1024)
RESIZE_RESOLUTION   = None     # Max dimension ceiling for input panorama (or None to keep original)
BATCH_SIZE          = 1        # Batch size for perspective view inference
SAVE_MAPS           = True     # Save depth_vis.png, depth.exr, and mask.png
SAVE_POINTS_PLY     = True     # Save 3D point cloud pointcloud.ply
SAVE_DEBUG          = False    # Save 12 individual splitted perspective views and camera JSONs

# ---- Kaggle Dataset Upload Configuration (Optional) ----
KAGGLE_USERNAME = "newmailserver"
DATASET_SLUG    = "marigold-pano-output"
UPLOAD_DIR      = "/kaggle/temp/upload"   # staging dir containing output + metadata

# Create base directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Config loaded successfully.")
print("Input dir:   ", INPUT_DIR)
print("Output dir:  ", OUTPUT_DIR)
print("Repo:        ", MARIGOLD_REPO)
print("Checkpoint:  ", MARIGOLD_CHECKPOINT)
print("Base Model:  ", BASE_MODEL)
print("Quantization:", QUANTIZATION)


In [ ]:
# =============================================================================
# 2. INSTALLATION (UV SYNC FROM PYPROJECT.TOML)
# =============================================================================
import os

os.environ["MPLBACKEND"] = "Agg"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

!pip install -q uv

%cd {WORK_DIR}/..
!rm -rf {WORK_DIR}
!git clone --depth 1 {MARIGOLD_REPO} {WORK_DIR}

%cd {WORK_DIR}
!uv sync

# Verify installation via uv virtual environment
!uv run python -c "import torch, diffusers, cv2, standalone_marigold; print('All Marigold V2 modules and dependencies imported successfully via uv!')"

# Post-processing & KaggleHub dependencies in base environment
!pip install -q opencv-python Pillow numpy imageio kagglehub

print("\nInstall complete.")


In [ ]:
# =============================================================================
# 3. RUN MARIGOLD V2 PANORAMA INFERENCE
# =============================================================================
import os
import sys

%cd {WORK_DIR}

# Build CLI command flags
flags = []
if USE_FP16:
    flags.append("--fp16")
if SAVE_MAPS:
    flags.append("--maps")
if SAVE_POINTS_PLY:
    flags.append("--points_ply")
if SAVE_DEBUG:
    flags.append("--debug")
if RESIZE_RESOLUTION is not None:
    flags.extend(["--resize", str(RESIZE_RESOLUTION)])

flag_str = " ".join(flags)

# Run single CLI execution on entire input dataset folder (loads model ONCE onto GPU)
!uv run python app.py \
    -i "{INPUT_DIR}" \
    -o "{OUTPUT_DIR}" \
    -c "{MARIGOLD_CHECKPOINT}" \
    --base_model "{BASE_MODEL}" \
    -m "{MODALITY}" \
    -q "{QUANTIZATION}" \
    --device cuda \
    --split_resolution {SPLIT_RESOLUTION} \
    --batch_size {BATCH_SIZE} \
    {flag_str}


In [ ]:
# =============================================================================
# 4. D2P POST-PROCESSING (3D POINT CLOUDS & SURFACE NORMALS)
# =============================================================================
import os
import glob
from typing import Optional, Tuple, Union
import numpy as np
from PIL import Image
import cv2

def image_uv(width: int, height: int, left=None, top=None, right=None, bottom=None, dtype=np.float32) -> np.ndarray:
    if left is None: left = 0
    if top is None: top = 0
    if right is None: right = width
    if bottom is None: bottom = height
    u = np.linspace((left + 0.5) / width, (right - 0.5) / width, right - left, dtype=dtype)
    v = np.linspace((top + 0.5) / height, (bottom - 0.5) / height, bottom - top, dtype=dtype)
    u, v = np.meshgrid(u, v, indexing='xy')
    return np.stack([u, v], axis=2)

def sphere_uv2dirs(uv: np.ndarray) -> np.ndarray:
    theta = (1.0 - uv[..., 0]) * (2.0 * np.pi)
    phi = uv[..., 1] * np.pi
    directions = np.stack([
        np.sin(phi) * np.cos(theta),
        np.sin(phi) * np.sin(theta),
        np.cos(phi)
    ], axis=-1)
    return directions

def points_to_normals(point: np.ndarray, mask: Optional[np.ndarray] = None) -> Union[np.ndarray, Tuple[np.ndarray, np.ndarray]]:
    height, width = point.shape[-3:-1]
    has_mask = mask is not None
    if mask is None:
        mask = np.ones((height, width), dtype=bool)
    else:
        mask = mask.astype(bool)
    mask_pad = np.zeros((height + 2, width + 2), dtype=bool)
    mask_pad[1:-1, 1:-1] = mask
    pts = np.zeros((height + 2, width + 2, 3), dtype=point.dtype)
    pts[1:-1, 1:-1, :] = point
    up = pts[:-2, 1:-1, :] - pts[1:-1, 1:-1, :]
    down = pts[2:, 1:-1, :] - pts[1:-1, 1:-1, :]
    left = pts[1:-1, :-2, :] - pts[1:-1, 1:-1, :]
    right = pts[1:-1, 2:, :] - pts[1:-1, 1:-1, :]
    mask_up = mask_pad[:-2, 1:-1]
    mask_down = mask_pad[2:, 1:-1]
    mask_left = mask_pad[1:-1, :-2]
    mask_right = mask_pad[1:-1, 2:]
    cross1 = np.cross(left, up)
    cross2 = np.cross(up, right)
    cross3 = np.cross(right, down)
    cross4 = np.cross(down, left)
    normals = (
        cross1 * (mask_left & mask_up)[..., None] +
        cross2 * (mask_up & mask_right)[..., None] +
        cross3 * (mask_right & mask_down)[..., None] +
        cross4 * (mask_down & mask_left)[..., None]
    )
    normals = normals / np.linalg.norm(normals, axis=-1, keepdims=True).clip(min=1e-8)
    normals_mask = mask & ((mask_left & mask_up) | (mask_up & mask_right) | (mask_right & mask_down) | (mask_down & mask_left))
    if has_mask:
        return normals, normals_mask
    return normals

def depth2points(depth: np.ndarray, mask: Optional[np.ndarray] = None) -> Union[np.ndarray, Tuple[np.ndarray, np.ndarray]]:
    height, width = depth.shape[-2:]
    uv = image_uv(width=width, height=height, dtype=depth.dtype)
    directions = sphere_uv2dirs(uv)
    points = directions * depth[..., None]
    if mask is not None:
        return points, mask
    return points

def colorize_normals(normals: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
    norm_rgb = ((normals + 1.0) * 0.5 * 255.0).clip(0, 255).astype(np.uint8)
    if mask is not None:
        norm_rgb = np.where(mask[..., None], norm_rgb, 0)
    return norm_rgb

def export_ply(filename: str, points: np.ndarray, colors: Optional[np.ndarray] = None, normals: Optional[np.ndarray] = None, mask: Optional[np.ndarray] = None):
    pts = points.reshape(-1, 3)
    m = mask.reshape(-1) if mask is not None else np.ones(len(pts), dtype=bool)
    pts = pts[m]
    cols = colors.reshape(-1, 3)[m] if colors is not None else None
    norms = normals.reshape(-1, 3)[m] if normals is not None else None
    with open(filename, "w") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(pts)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        if norms is not None:
            f.write("property float nx\nproperty float ny\nproperty float nz\n")
        if cols is not None:
            f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        f.write("end_header\n")
        for i in range(len(pts)):
            line = f"{pts[i,0]:.4f} {pts[i,1]:.4f} {pts[i,2]:.4f}"
            if norms is not None:
                line += f" {norms[i,0]:.4f} {norms[i,1]:.4f} {norms[i,2]:.4f}"
            if cols is not None:
                line += f" {int(cols[i,0])} {int(cols[i,1])} {int(cols[i,2])}"
            f.write(line + "\n")

# Post-process all output subfolders
out_subdirs = sorted([d for d in glob.glob(os.path.join(OUTPUT_DIR, "**"), recursive=True) if os.path.isfile(os.path.join(d, "depth.npy"))])
print(f"Found {len(out_subdirs)} completed scenes for D2P post-processing.")

for folder in out_subdirs:
    d_npy = os.path.join(folder, "depth.npy")
    if not os.path.isfile(d_npy): continue
    depth = np.load(d_npy)
    m_path = os.path.join(folder, "mask.png")
    mask = (cv2.imread(m_path, cv2.IMREAD_GRAYSCALE) > 128) if os.path.isfile(m_path) else None
    pts, _ = depth2points(depth, mask=mask)
    np.save(os.path.join(folder, "points.npy"), pts)
    norms, nmask = points_to_normals(pts, mask=mask)
    norm_vis = colorize_normals(norms, mask=nmask)
    cv2.imwrite(os.path.join(folder, "normal_vis.png"), cv2.cvtColor(norm_vis, cv2.COLOR_RGB2BGR))
    # Find matching original image
    rel = os.path.relpath(folder, OUTPUT_DIR)
    for ext in (".jpg", ".png", ".jpeg", ".webp"):
        orig_p = os.path.join(INPUT_DIR, rel + ext)
        if os.path.isfile(orig_p):
            rgb = cv2.cvtColor(cv2.imread(orig_p), cv2.COLOR_BGR2RGB)
            if rgb.shape[:2] != depth.shape[:2]:
                rgb = cv2.resize(rgb, (depth.shape[1], depth.shape[0]), interpolation=cv2.INTER_AREA)
            export_ply(os.path.join(folder, "custom_d2p.ply"), pts, colors=rgb, normals=norms, mask=mask)
            break
    print(f"  ✅ Post-processed: {rel}")

print("✨ All D2P 3D reconstruction outputs generated!")


In [ ]:
# =============================================================================
# 5. KAGGLEHUB DATASET EXPORT & UPLOAD
# =============================================================================
import json
import shutil
import os
import kagglehub

ENABLE_UPLOAD = False   # Set to True when ready to export to Kaggle Datasets

if ENABLE_UPLOAD:
    print(f"Staging results from {OUTPUT_DIR} to {UPLOAD_DIR}...")
    if os.path.exists(UPLOAD_DIR):
        shutil.rmtree(UPLOAD_DIR)
    shutil.copytree(OUTPUT_DIR, UPLOAD_DIR)

    dataset_meta = {
        "title": "Marigold 360 Panorama Depth Output",
        "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
        json.dump(dataset_meta, f, indent=2)

    print(f"Uploading dataset to {KAGGLE_USERNAME}/{DATASET_SLUG} via KaggleHub...")
    kagglehub.dataset_upload(
        handle=f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
        local_dataset_dir=UPLOAD_DIR,
        version_notes="Marigold 360 Depth & Point Cloud Generation"
    )
    print("🎉 Kaggle dataset uploaded successfully!")
else:
    print("Dataset upload skipped (ENABLE_UPLOAD = False).")
